# Predicting Electric Vehicle Purchases
### Kaggle Playground Series S6E9 — binary classification, metric: ROC-AUC

## 1. Project Overview

| | |
|---|---|
| **Competition** | Kaggle Playground Series, Season 6 Episode 9 |
| **Objective** | Predict whether a consumer will buy an electric vehicle |
| **Target** | `Will_Buy_EV` (`Yes` / `No`); submission needs the probability of `Yes` |
| **Metric** | ROC-AUC |
| **Data** | `train.csv` (668,665 rows, 15 columns), `test.csv` (286,571 rows, 14 columns); demographic, commute, charging-infrastructure and attitude features |

**Approach.** Audit and validate the data, engineer domain features and keep only those that survive a validation ablation, build a logistic-regression baseline, then train LightGBM, XGBoost, CatBoost and ExtraTrees under one shared 5-fold stratified CV. Boosters get a targeted random search. Blend weights are fitted on out-of-fold (OOF) predictions only, and the final models are refit on all training rows.

**Reporting convention.** Every score in this notebook is computed when it runs; nothing is hard-coded. *CV ROC-AUC* (OOF on train) is kept strictly separate from *Kaggle public LB ROC-AUC*, which only exists once `submission.csv` has been uploaded.

## 2. Imports and Configuration

In [ ]:
!pip install catboost lightgbm xgboost --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.optimize import minimize

In [ ]:
import os
import subprocess
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import minimize
from scipy.special import logit, softmax
from scipy.stats import loguniform, randint, rankdata, uniform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import ParameterSampler, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

try:
    import catboost  # noqa: F401  (not preinstalled on Colab)
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.5f}".format)
sns.set_theme(style="whitegrid")

SEED = 42
N_FOLDS = 5
ID_COL, TARGET_COL = "id", "Will_Buy_EV"
USE_GPU = False          # set True on a GPU runtime to speed up XGBoost / CatBoost
np.random.seed(SEED)

## 3. Data Loading

In [ ]:
DATA_DIR = next(
    (d for d in ["/content", "/kaggle/input/playground-series-s6e9", ".", "data"]
     if os.path.exists(os.path.join(d, "train.csv"))),
    None,
)
assert DATA_DIR is not None, "train.csv not found - upload the competition files first."

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"train_df: {train_df.shape} | test_df: {test_df.shape}")
assert TARGET_COL in train_df.columns and TARGET_COL not in test_df.columns
assert ID_COL in train_df.columns and ID_COL in test_df.columns
assert list(train_df.columns.drop(TARGET_COL)) == list(test_df.columns), "train/test columns differ"
train_df.head()

## 4. Data Cleaning

The same audit runs on train and test. It only *reports*; rows are removed only if a check fails and is investigated first. Both files are read with default `pd.read_csv` (no `on_bad_lines`), so no row is silently dropped at load time.

In [ ]:
def audit_frame(df, name):
    feature_cols = [c for c in df.columns if c not in (ID_COL, TARGET_COL)]
    invalid = {
        "Age < 18": (df["Age"] < 18).sum(),
        "Annual_Income_USD <= 0": (df["Annual_Income_USD"] <= 0).sum(),
        "Daily_Commute_km < 0": (df["Daily_Commute_km"] < 0).sum(),
        "Number_of_Cars_Owned < 0": (df["Number_of_Cars_Owned"] < 0).sum(),
        "Charging_Stations < 0": ((df["Charging_Stations_Near_Home"] < 0) | (df["Charging_Stations_Near_Work"] < 0)).sum(),
        "Environmental_Concern_Level outside 1-5": (~df["Environmental_Concern_Level"].between(1, 5)).sum(),
    }
    numeric_ok = all(pd.api.types.is_numeric_dtype(df[c]) for c in
                     ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
                      "Charging_Stations_Near_Home", "Charging_Stations_Near_Work", "Environmental_Concern_Level"])
    report = {
        "rows": len(df),
        "missing cells": int(df.isna().sum().sum()),
        "duplicate ids": int(df[ID_COL].duplicated().sum()),
        "duplicate feature rows (kept)": int(df.duplicated(subset=feature_cols).sum()),
        "constant columns": [c for c in feature_cols if df[c].nunique(dropna=False) == 1],
        "numeric columns have numeric dtype": numeric_ok,
        **{f"invalid: {k}": int(v) for k, v in invalid.items()},
    }
    print(f"--- {name} ---")
    for k, v in report.items():
        print(f"{k:45s} {v}")
    assert report["missing cells"] == 0 and report["duplicate ids"] == 0 and numeric_ok
    assert not report["constant columns"] and all(int(v) == 0 for v in invalid.values())

### Train cleaning

In [ ]:
audit_frame(train_df, "train_df")
assert set(train_df[TARGET_COL].unique()) == {"Yes", "No"}
y = train_df[TARGET_COL].map({"No": 0, "Yes": 1}).to_numpy()   # target kept separately; train_df is unchanged
assert not np.isnan(y.astype(float)).any()

### Test cleaning

In [ ]:
audit_frame(test_df, "test_df")
train_levels = {c: set(train_df[c].unique()) for c in train_df.select_dtypes(exclude="number").columns.drop(TARGET_COL)}
for c, levels in train_levels.items():
    assert set(test_df[c].unique()) <= levels, f"unseen category in test column {c}"
print("\nAll categorical levels in test_df also appear in train_df. No cleaning transformations were required.")

## 5. Target Analysis

In [ ]:
target_summary = pd.DataFrame({
    "count": train_df[TARGET_COL].value_counts(),
    "percent": train_df[TARGET_COL].value_counts(normalize=True) * 100,
})
display(target_summary)
print(f"Imbalance ratio (No : Yes) = {target_summary.loc['No', 'count'] / target_summary.loc['Yes', 'count']:.2f} : 1")

fig, ax = plt.subplots(figsize=(4.5, 3.5))
sns.barplot(x=target_summary.index, y=target_summary["count"], ax=ax, palette=["#8fa8c8", "#e0864f"])
for i, (cnt, pct) in enumerate(zip(target_summary["count"], target_summary["percent"])):
    ax.text(i, cnt, f"{pct:.1f}%", ha="center", va="bottom")
ax.set_title("Will_Buy_EV class balance"); ax.set_xlabel(""); plt.tight_layout(); plt.show()

The classes are imbalanced. ROC-AUC is rank-based and insensitive to the class ratio, so no resampling or class weighting is used; stratified folds keep the ratio identical across folds.

## 6. Missing Values / Imputation

In [ ]:
missing_train, missing_test = int(train_df.isna().sum().sum()), int(test_df.isna().sum().sum())
print(f"Missing cells - train: {missing_train}, test: {missing_test}")
assert missing_train == 0 and missing_test == 0

There are **no missing values** in either file, so **no imputer is used**. Adding one would only add an untested transformation. If a future data version contains NaNs, fit a `SimpleImputer` inside each CV fold's pipeline.

## 7. Feature Types

In [ ]:
numerical_features = [
    "Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home", "Charging_Stations_Near_Work", "Environmental_Concern_Level",
]
ordinal_features = {"Range_Anxiety_Level": ["Low", "Medium", "High"]}   # ordered categories
categorical_features = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible", "Subsidy_Available"]

raw_feature_cols = [c for c in train_df.columns if c not in (ID_COL, TARGET_COL)]
assert sorted(numerical_features + list(ordinal_features) + categorical_features) == sorted(raw_feature_cols)
print(f"id: {ID_COL} | target: {TARGET_COL}")
print(f"numerical ({len(numerical_features)}), ordinal ({len(ordinal_features)}), nominal categorical ({len(categorical_features)})")

## 8. Feature Engineering

Features carried over from the earlier notebook are grouped into six families. All are row-wise transformations, so none depends on the target and none can leak across folds. Two fixes from the audit: `Range_Anxiety_Level` is now encoded in its true order (Low < Medium < High) instead of alphabetically, and age buckets use fixed edges instead of `pd.cut(bins=n)`, which would derive different edges on train and test.

Whether the engineered features help is decided by a LightGBM **ablation** on a stratified subsample: score the base features, base + all engineered features, and each family removed in turn. A family is kept only if removing it lowers AUC by more than a small noise floor.

In [ ]:
RANGE_ANXIETY_ORDER = {level: i for i, level in enumerate(ordinal_features["Range_Anxiety_Level"])}
AGE_BIN_EDGES = [0, 34, 44, 54, 64, 120]
CAT_DTYPES = {c: pd.CategoricalDtype(sorted(train_df[c].unique())) for c in categorical_features}

FEATURE_FAMILIES = {
    "income": ["Log_Annual_Income", "Income_per_Car", "Income_to_Commute_Ratio"],
    "charging_infrastructure": ["Total_Charging_Stations", "Stations_per_Car", "Any_Charging_Available"],
    "range_anxiety": ["Commute_x_RangeAnxiety", "Anxiety_vs_Charging_Density"],
    "concern": ["Concern_per_Commute", "Concern_x_TotalStations", "Age_x_Concern"],
    "convenience": ["Subsidy_and_HomeCharge", "Convenience_Score"],
    "age": ["Age_bucket"],
}


def engineer_features(df):
    out = df.copy()
    out["Range_Anxiety_Numeric"] = out["Range_Anxiety_Level"].map(RANGE_ANXIETY_ORDER)
    home = (out["Home_Charging_Possible"] == "Yes").astype(int)
    subsidy = (out["Subsidy_Available"] == "Yes").astype(int)
    total = out["Charging_Stations_Near_Home"] + out["Charging_Stations_Near_Work"]
    cars = out["Number_of_Cars_Owned"].clip(lower=1)
    commute = out["Daily_Commute_km"].clip(lower=1)

    out["Log_Annual_Income"] = np.log1p(out["Annual_Income_USD"])
    out["Income_per_Car"] = out["Annual_Income_USD"] / cars
    out["Income_to_Commute_Ratio"] = out["Annual_Income_USD"] / commute
    out["Total_Charging_Stations"] = total
    out["Stations_per_Car"] = total / cars
    out["Any_Charging_Available"] = (total > 0).astype(int)
    out["Commute_x_RangeAnxiety"] = out["Range_Anxiety_Numeric"] * out["Daily_Commute_km"]
    out["Anxiety_vs_Charging_Density"] = out["Range_Anxiety_Numeric"] / (total + 1)
    out["Concern_per_Commute"] = out["Environmental_Concern_Level"] / commute
    out["Concern_x_TotalStations"] = out["Environmental_Concern_Level"] * total
    out["Age_x_Concern"] = out["Age"] * out["Environmental_Concern_Level"]
    out["Subsidy_and_HomeCharge"] = home * subsidy
    out["Convenience_Score"] = home + 0.5 * out["Charging_Stations_Near_Work"] + subsidy
    out["Age_bucket"] = pd.cut(out["Age"], bins=AGE_BIN_EDGES, labels=False).astype(int)

    for c in categorical_features:                      # same category set for train and test
        out[c] = out[c].astype(CAT_DTYPES[c])
    return out


train_fe, test_fe = engineer_features(train_df), engineer_features(test_df)
engineered_cols = [f for fam in FEATURE_FAMILIES.values() for f in fam]
assert not train_fe[engineered_cols + ["Range_Anxiety_Numeric"]].isna().any().any()
assert not test_fe[engineered_cols + ["Range_Anxiety_Numeric"]].isna().any().any()
assert np.isfinite(train_fe[engineered_cols].to_numpy(dtype=float)).all()

base_features = numerical_features + ["Range_Anxiety_Numeric"] + categorical_features
all_features = base_features + engineered_cols
print(f"base features: {len(base_features)} | engineered candidates: {len(engineered_cols)}")

In [ ]:
# ---- Feature-family ablation (LightGBM, stratified subsample, 3-fold) ----
ABLATION_ROWS, ABLATION_NOISE_FLOOR = 300_000, 2e-5


def stratified_subsample(n_rows, seed=SEED):
    if n_rows >= len(y):
        return np.arange(len(y))
    return train_test_split(np.arange(len(y)), train_size=n_rows, stratify=y, random_state=seed)[0]


abl_idx = stratified_subsample(ABLATION_ROWS)
X_abl, y_abl = train_fe.iloc[abl_idx].reset_index(drop=True), y[abl_idx]
abl_folds = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(X_abl, y_abl))
QUICK_PARAMS = dict(objective="binary", metric="auc", learning_rate=0.1, num_leaves=63, min_child_samples=50,
                    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, verbosity=-1, seed=SEED)


def quick_cv_auc(feature_list):
    oof = np.zeros(len(y_abl))
    for tr, va in abl_folds:
        d_tr = lgb.Dataset(X_abl.iloc[tr][feature_list], y_abl[tr])
        d_va = lgb.Dataset(X_abl.iloc[va][feature_list], y_abl[va], reference=d_tr)
        booster = lgb.train(QUICK_PARAMS, d_tr, 1000, valid_sets=[d_va],
                            callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va] = booster.predict(X_abl.iloc[va][feature_list], num_iteration=booster.best_iteration)
    return roc_auc_score(y_abl, oof)


ablation = {"base only": quick_cv_auc(base_features), "base + all engineered": quick_cv_auc(all_features)}
for fam, cols in FEATURE_FAMILIES.items():
    ablation[f"all - {fam}"] = quick_cv_auc([f for f in all_features if f not in cols])

kept_families = [fam for fam in FEATURE_FAMILIES
                 if ablation["base + all engineered"] - ablation[f"all - {fam}"] > ABLATION_NOISE_FLOOR]
selected_features = base_features + [f for fam in kept_families for f in FEATURE_FAMILIES[fam]]
ablation["selected families"] = quick_cv_auc(selected_features)

candidate_sets = {"base only": base_features, "base + all engineered": all_features, "selected families": selected_features}
best_set = max(candidate_sets, key=lambda k: (round(ablation[k], 6), -len(candidate_sets[k])))
features = candidate_sets[best_set]

display(pd.Series(ablation, name="ablation ROC-AUC").to_frame())
print(f"Kept families: {kept_families}")
print(f"Feature set used from here on: '{best_set}' ({len(features)} features)")

In [ ]:
X, X_test = train_fe[features], test_fe[features]
cat_features = [c for c in categorical_features if c in features]
print(f"X: {X.shape} | X_test: {X_test.shape}")

## 9. Scaling

LightGBM, XGBoost, CatBoost and ExtraTrees split on feature thresholds, so they are invariant to monotonic rescaling. **They are trained on unscaled features.** Scaling is applied only where it matters: the logistic-regression baseline, inside a `Pipeline`, so its mean and variance are learned from each training fold only.

In [ ]:
baseline_features = numerical_features + list(ordinal_features) + categorical_features
baseline_model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), numerical_features),
        ("ord", Pipeline([("enc", OrdinalEncoder(categories=list(ordinal_features.values()))),
                          ("scale", StandardScaler())]), list(ordinal_features)),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])),
    ("clf", LogisticRegression(max_iter=1000)),
])

## 10. Statistical Analysis

In [ ]:
display(train_df[numerical_features].describe().T[["mean", "std", "min", "50%", "max"]])

card = pd.DataFrame({"train": train_df[categorical_features + list(ordinal_features)].nunique(),
                     "test": test_df[categorical_features + list(ordinal_features)].nunique()})
display(card.rename_axis("categorical cardinality"))

# Train/test shift: standardized difference of means (values near 0 mean matching distributions)
shift = ((test_df[numerical_features].mean() - train_df[numerical_features].mean()) / train_df[numerical_features].std())
display(shift.rename("standardized mean shift (test - train)").to_frame())

# Target relationship
target_rates = pd.concat([train_df.groupby(c)[TARGET_COL].apply(lambda s: (s == "Yes").mean()).rename_axis("level")
                          .reset_index().assign(feature=c) for c in categorical_features + list(ordinal_features)])
display(target_rates.set_index(["feature", "level"]).rename(columns={TARGET_COL: "P(Will_Buy_EV = Yes)"}))
corr_with_target = train_fe[numerical_features + ["Range_Anxiety_Numeric"]].corrwith(pd.Series(y)).sort_values()
display(corr_with_target.rename("correlation with target").to_frame())

## 11. Visualization / EDA

In [ ]:
plot_idx = stratified_subsample(100_000, seed=1)                 # subsample keeps plotting fast
plot_df, plot_y = train_fe.iloc[plot_idx], pd.Series(y[plot_idx], name="Will_Buy_EV")

# Q1: which numerical variables separate buyers from non-buyers?
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, col in zip(axes.ravel(), ["Age", "Annual_Income_USD", "Daily_Commute_km",
                                  "Charging_Stations_Near_Home", "Charging_Stations_Near_Work", "Environmental_Concern_Level"]):
    sns.histplot(x=plot_df[col], hue=plot_y.map({0: "No", 1: "Yes"}).values, stat="density", common_norm=False,
                 element="step", fill=False, bins=30, ax=ax, legend=(col == "Age"))
    ax.set_title(col)
plt.suptitle("Numerical distributions by target", y=1.02); plt.tight_layout(); plt.show()

# Q2: how does purchase rate vary across categories?
cat_cols_plot = categorical_features + list(ordinal_features)
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for ax, col in zip(axes.ravel(), cat_cols_plot):
    rate = train_df.groupby(col)[TARGET_COL].apply(lambda s: (s == "Yes").mean())
    if col in ordinal_features:
        rate = rate.reindex(ordinal_features[col])
    sns.barplot(x=rate.index.astype(str), y=rate.values, ax=ax, color="#5b8fb9")
    ax.axhline(y.mean(), color="grey", ls="--", lw=1); ax.set_title(col); ax.set_ylabel("P(Yes)")
plt.suptitle("Purchase rate by category (dashed = overall rate)", y=1.02); plt.tight_layout(); plt.show()

# Q3: which numerical features are linearly related to each other and the target?
corr_frame = train_fe[numerical_features + ["Range_Anxiety_Numeric"]].assign(Will_Buy_EV=y)
plt.figure(figsize=(9, 6.5))
sns.heatmap(corr_frame.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.4)
plt.title("Correlation matrix (numerical + ordinal + target)"); plt.tight_layout(); plt.show()

## 12. Cross-Validation

One set of stratified 5-fold splits (`folds`) is created once and reused by every model, so all OOF predictions are directly comparable and can be blended. Everything fitted on data (scaler, encoders, models, early stopping) sees only the training part of each fold. Feature engineering is target-free, and no target encoding is used.

Two caveats are stated up front. LightGBM/XGBoost/CatBoost use each fold's validation part for early stopping, which is standard practice but makes the OOF scores very slightly optimistic. Hyperparameters are chosen with CV on the training data, which adds a small selection effect. Neither uses the test set.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
print(pd.DataFrame({"train rows": [len(tr) for tr, _ in folds], "valid rows": [len(va) for _, va in folds],
                    "valid positive rate": [y[va].mean() for _, va in folds]}).rename_axis("fold"))

cv_results = {}   # name -> dict(oof, test_avg, fold_aucs, oof_auc, best_iters, params, family)


def register(name, family, oof, test_avg=None, best_iters=None, params=None):
    fold_aucs = [roc_auc_score(y[va], oof[va]) for _, va in folds]
    cv_results[name] = dict(family=family, oof=oof, test_avg=test_avg, best_iters=best_iters, params=params,
                            fold_aucs=fold_aucs, oof_auc=roc_auc_score(y, oof))
    r = cv_results[name]
    print(f"{name:14s} OOF ROC-AUC = {r['oof_auc']:.6f} | fold mean = {np.mean(fold_aucs):.6f} +/- {np.std(fold_aucs):.6f}")

## 13. Baseline Model

Logistic regression on the raw features (scaled and one-hot encoded inside the pipeline) sets the reference score. It is fitted from scratch in every fold.

In [ ]:
oof_base = np.zeros(len(y))
for tr, va in folds:
    baseline_model.fit(train_df.iloc[tr][baseline_features], y[tr])
    oof_base[va] = baseline_model.predict_proba(train_df.iloc[va][baseline_features])[:, 1]
register("baseline_lr", "linear", oof_base)
baseline_auc = cv_results["baseline_lr"]["oof_auc"]

## 14. Advanced Models

LightGBM, XGBoost and CatBoost are trained on the selected features with reasonable starting hyperparameters. LightGBM gets pandas categoricals, XGBoost and ExtraTrees get integer category codes, and CatBoost gets string categoricals with its native handling. **ExtraTrees** is a bagging-style model that may add prediction diversity; it is kept only if the ensemble step finds it useful.

All model code lives in a few small functions so the same code serves the default runs, the tuning search and the final refit.

In [ ]:
# ---- Model matrices: one representation per model family ----
X_codes, X_test_codes = X.copy(), X_test.copy()
X_str, X_test_str = X.copy(), X_test.copy()
for c in cat_features:
    X_codes[c], X_test_codes[c] = X[c].cat.codes.astype("int16"), X_test[c].cat.codes.astype("int16")
    X_str[c], X_test_str[c] = X[c].astype(str), X_test[c].astype(str)
MATRICES = {"lgb": (X, X_test), "xgb": (X_codes, X_test_codes), "cat": (X_str, X_test_str), "et": (X_codes, X_test_codes)}

ROUND_CAP = {"lgb": 5000, "xgb": 5000, "cat": 3000, "et": None}
LGB_FIXED = dict(objective="binary", metric="auc", bagging_freq=1, verbosity=-1)
XGB_FIXED = dict(tree_method="hist", eval_metric="auc", n_jobs=-1, verbosity=0, **({"device": "cuda"} if USE_GPU else {}))
CAT_FIXED = dict(eval_metric="AUC", bootstrap_type="Bayesian", verbose=False, thread_count=-1, allow_writing_files=False,
                 **({"task_type": "GPU"} if USE_GPU else {}))
ET_FIXED = dict(n_estimators=120, min_samples_leaf=30, max_features=0.6, bootstrap=True, max_samples=0.5, n_jobs=-1)


def fit_predict_lgb(params, X_tr, y_tr, X_va, y_va, X_new, seed, rounds):
    d_tr = lgb.Dataset(X_tr, y_tr)
    d_va = lgb.Dataset(X_va, y_va, reference=d_tr)
    booster = lgb.train({**LGB_FIXED, **params, "seed": seed}, d_tr, rounds, valid_sets=[d_va],
                        callbacks=[lgb.early_stopping(100, verbose=False)])
    it = booster.best_iteration
    return (booster.predict(X_va, num_iteration=it),
            None if X_new is None else booster.predict(X_new, num_iteration=it), it)


def fit_predict_xgb(params, X_tr, y_tr, X_va, y_va, X_new, seed, rounds):
    model = xgb.XGBClassifier(**{**XGB_FIXED, **params}, n_estimators=rounds, early_stopping_rounds=100, random_state=seed)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    it = model.best_iteration + 1
    return (model.predict_proba(X_va, iteration_range=(0, it))[:, 1],
            None if X_new is None else model.predict_proba(X_new, iteration_range=(0, it))[:, 1], it)


def fit_predict_cat(params, X_tr, y_tr, X_va, y_va, X_new, seed, rounds):
    model = CatBoostClassifier(**{**CAT_FIXED, **params}, iterations=rounds, early_stopping_rounds=100, random_seed=seed)
    model.fit(Pool(X_tr, y_tr, cat_features=cat_features), eval_set=Pool(X_va, y_va, cat_features=cat_features))
    it = model.get_best_iteration() + 1
    return (model.predict_proba(Pool(X_va, cat_features=cat_features))[:, 1],
            None if X_new is None else model.predict_proba(Pool(X_new, cat_features=cat_features))[:, 1], it)


def fit_predict_et(params, X_tr, y_tr, X_va, y_va, X_new, seed, rounds):
    model = ExtraTreesClassifier(**{**ET_FIXED, **params}, random_state=seed).fit(X_tr, y_tr)
    return model.predict_proba(X_va)[:, 1], None if X_new is None else model.predict_proba(X_new)[:, 1], None


FIT_PREDICT = {"lgb": fit_predict_lgb, "xgb": fit_predict_xgb, "cat": fit_predict_cat, "et": fit_predict_et}


def run_cv(family, params, X_data, y_data, X_new, folds_, rounds=None):
    """Generic K-fold loop. Returns OOF predictions, fold-averaged predictions for X_new, best iterations."""
    rounds = rounds or ROUND_CAP[family]
    oof = np.zeros(len(y_data))
    new_avg = None if X_new is None else np.zeros(len(X_new))
    iters = []
    for k, (tr, va) in enumerate(folds_):
        p_va, p_new, it = FIT_PREDICT[family](params, X_data.iloc[tr], y_data[tr], X_data.iloc[va], y_data[va],
                                              X_new, SEED + k, rounds)
        oof[va] = p_va
        if X_new is not None:
            new_avg += p_new / len(folds_)
        iters.append(it)
    return oof, new_avg, iters


DEFAULT_PARAMS = {
    "lgb": dict(learning_rate=0.05, num_leaves=127, max_depth=8, feature_fraction=0.8, bagging_fraction=0.8, min_child_samples=30),
    "xgb": dict(learning_rate=0.05, max_depth=7, subsample=0.8, colsample_bytree=0.8, min_child_weight=3),
    "cat": dict(learning_rate=0.05, depth=7, l2_leaf_reg=5),
    "et": {},
}
for family in ["lgb", "xgb", "cat", "et"]:
    X_data, X_new = MATRICES[family]
    oof, new_avg, iters = run_cv(family, DEFAULT_PARAMS[family], X_data, y, X_new, folds)
    register(f"{family}_default", family, oof, new_avg, iters, DEFAULT_PARAMS[family])

## 15. Model Optimization

A targeted random search (not a grid) runs on a stratified subsample with 3-fold CV and a higher learning rate for speed. The best configuration per booster is then re-run on the full 5-fold splits at a lower learning rate. Search ranges cover the parameters listed in the project brief. `TUNING` controls the budget: reduce `trials` for a quicker run or increase it if time allows.

In [ ]:
TUNE_ROWS, TUNE_FOLDS = 250_000, 3
TUNING = {
    "lgb": dict(trials=12, lr=0.08, final_lr=0.03, rounds=2000, space=dict(
        num_leaves=randint(31, 256), max_depth=[6, 8, 10, 12], min_child_samples=randint(10, 150),
        feature_fraction=uniform(0.5, 0.5), bagging_fraction=uniform(0.6, 0.4),
        lambda_l1=loguniform(1e-3, 10), lambda_l2=loguniform(1e-3, 30))),
    "xgb": dict(trials=10, lr=0.08, final_lr=0.03, rounds=2000, space=dict(
        max_depth=randint(4, 11), min_child_weight=loguniform(1, 50), subsample=uniform(0.6, 0.4),
        colsample_bytree=uniform(0.5, 0.5), reg_alpha=loguniform(1e-3, 10), reg_lambda=loguniform(0.1, 30),
        gamma=loguniform(1e-3, 5))),
    "cat": dict(trials=8, lr=0.1, final_lr=0.05, rounds=1000, space=dict(
        depth=randint(5, 10), l2_leaf_reg=loguniform(1, 30), random_strength=loguniform(0.1, 10),
        bagging_temperature=uniform(0, 3))),
}
tune_idx = stratified_subsample(TUNE_ROWS, seed=SEED + 1)
y_tune = y[tune_idx]
tune_folds = list(StratifiedKFold(TUNE_FOLDS, shuffle=True, random_state=SEED + 1).split(tune_idx, y_tune))


def to_native(d):
    return {k: (v.item() if hasattr(v, "item") else v) for k, v in d.items()}


tuning_logs, best_params = {}, {}
for family, cfg in TUNING.items():
    X_tune = MATRICES[family][0].iloc[tune_idx].reset_index(drop=True)
    rows, sampled_params = [], []
    for sampled in ParameterSampler(cfg["space"], n_iter=cfg["trials"], random_state=SEED):
        sampled = to_native(sampled)             # plain Python ints/floats (a DataFrame row would turn ints into floats)
        oof_t, _, iters_t = run_cv(family, {**sampled, "learning_rate": cfg["lr"]}, X_tune, y_tune, None, tune_folds, cfg["rounds"])
        sampled_params.append(sampled)
        rows.append({**sampled, "AUC": roc_auc_score(y_tune, oof_t), "mean_best_iter": np.mean(iters_t)})
    tuning_logs[family] = pd.DataFrame(rows).sort_values("AUC", ascending=False)
    best_params[family] = sampled_params[int(np.argmax([r["AUC"] for r in rows]))]
    print(f"\n[{family}] top search trials (subsample AUC, not comparable to full-data OOF):")
    display(tuning_logs[family].head(3))

    tuned = {**best_params[family], "learning_rate": cfg["final_lr"]}
    X_data, X_new = MATRICES[family]
    oof, new_avg, iters = run_cv(family, tuned, X_data, y, X_new, folds)
    register(f"{family}_tuned", family, oof, new_avg, iters, tuned)


[lgb] top search trials (subsample AUC, not comparable to full-data OOF):


,bagging_fraction,feature_fraction,lambda_l1,lambda_l2,max_depth,min_child_samples,num_leaves,AUC,mean_best_iter
2,0.88880,0.96928,0.00101,27.68544,6,67,52,0.94016,264.33333
10,0.93150,0.67838,0.01330,0.26898,6,24,75,0.94013,169.66667
8,0.81868,0.59243,7.55681,2.95368,8,23,125,0.94011,150.66667


lgb_tuned      OOF ROC-AUC = 0.941859 | fold mean = 0.941868 +/- 0.000718

[xgb] top search trials (subsample AUC, not comparable to full-data OOF):


,colsample_bytree,gamma,max_depth,min_child_weight,reg_alpha,reg_lambda,subsample,AUC,mean_best_iter
0,0.68727,3.28597,6,21.11892,0.24401,1.27169,0.63999,0.94034,715.33333
5,0.97110,0.12122,5,3.29253,0.00246,4.95368,0.77606,0.94017,255.00000
2,0.59091,0.00477,7,10.94411,0.00107,0.11406,0.80991,0.94010,118.33333


xgb_tuned      OOF ROC-AUC = 0.941913 | fold mean = 0.941922 +/- 0.000802


## 16. OOF Evaluation

Per model: pooled OOF ROC-AUC, mean and standard deviation of the five fold AUCs. For each booster the better of *default* and *tuned* (by OOF AUC) goes forward; ExtraTrees goes forward as a blend candidate.

In [ ]:
eval_table = pd.DataFrame({
    n: {"OOF ROC-AUC": r["oof_auc"], "Mean fold AUC": np.mean(r["fold_aucs"]), "Std fold AUC": np.std(r["fold_aucs"])}
    for n, r in cv_results.items()
}).T.sort_values("OOF ROC-AUC", ascending=False)
display(eval_table)

selected = {f: max([n for n in cv_results if cv_results[n]["family"] == f], key=lambda n: cv_results[n]["oof_auc"])
            for f in ["lgb", "xgb", "cat", "et"]}
print("Carried forward:", selected)
display(pd.DataFrame({n: cv_results[n]["fold_aucs"] for n in selected.values()},
                     index=[f"fold {i + 1}" for i in range(N_FOLDS)]))

## 17. Ensemble

Everything here uses **OOF predictions only**; test predictions play no part in choosing weights.

1. **Diversity check:** Spearman correlation between the models' OOF predictions.
2. **Candidates:** best single model, simple mean, rank mean, non-negative weights that maximize OOF AUC (Nelder-Mead on a softmax parametrization), and a logistic-regression stacker on logit-transformed predictions.
3. **Honest scoring:** fitted candidates (weights, stacker) are scored with a second-level 5-fold split (fit on four OOF folds, score on the fifth), so a candidate is not rewarded for fitting its own evaluation data.
4. **Pruning:** models whose optimized weight is below `MIN_WEIGHT` are dropped and the candidates re-evaluated. The simplest candidate within `TIE_TOLERANCE` of the best score is chosen.

In [ ]:
MIN_WEIGHT, TIE_TOLERANCE = 0.02, 1e-5
members = list(selected.values())
display(pd.DataFrame({n: cv_results[n]["oof"] for n in members}).corr(method="spearman").rename_axis("Spearman corr. of OOF preds"))

meta_folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + 7).split(np.zeros(len(y)), y))


def rank01(a):
    return rankdata(a) / len(a)


def optimize_weights(P, y_):
    loss = lambda z: -roc_auc_score(y_, P @ softmax(z))
    res = minimize(loss, np.zeros(P.shape[1]), method="Nelder-Mead", options=dict(xatol=1e-3, fatol=1e-8, maxiter=200))
    return softmax(res.x)


def evaluate_blends(names):
    P = np.column_stack([cv_results[n]["oof"] for n in names])
    R = np.column_stack([rank01(P[:, j]) for j in range(P.shape[1])])
    Z = logit(np.clip(P, 1e-6, 1 - 1e-6))
    best_single = max(names, key=lambda n: cv_results[n]["oof_auc"])
    out = dict(names=names, best_single=best_single, weights=None, meta=None,
               scores={"single": cv_results[best_single]["oof_auc"]})
    if len(names) > 1:
        out["scores"]["mean"] = roc_auc_score(y, P.mean(1))
        out["scores"]["rank_mean"] = roc_auc_score(y, R.mean(1))
        nested_w, nested_s = np.zeros(len(y)), np.zeros(len(y))
        for tr, va in meta_folds:
            nested_w[va] = P[va] @ optimize_weights(P[tr], y[tr])
            nested_s[va] = LogisticRegression(max_iter=1000).fit(Z[tr], y[tr]).predict_proba(Z[va])[:, 1]
        out["scores"]["weighted"] = roc_auc_score(y, nested_w)
        out["scores"]["stack"] = roc_auc_score(y, nested_s)
        out["weights"] = optimize_weights(P, y)
        out["meta"] = LogisticRegression(max_iter=1000).fit(Z, y)
    return out


first_pass = evaluate_blends(members)
display(pd.Series(first_pass["scores"], name="honest OOF ROC-AUC (first pass)").to_frame())
print("Optimized weights:", {n: round(float(w), 4) for n, w in zip(members, first_pass["weights"])})

keep = [n for n, w in zip(members, first_pass["weights"]) if w >= MIN_WEIGHT]
dropped = [n for n in members if n not in keep]
final_eval = first_pass if not dropped else evaluate_blends(keep)
final_members = final_eval["names"]
if dropped:
    print(f"Dropped for negligible weight: {dropped}")
    display(pd.Series(final_eval["scores"], name="honest OOF ROC-AUC (pruned)").to_frame())

order = ["single", "mean", "rank_mean", "weighted", "stack"]
top = max(final_eval["scores"].values())
ensemble_method = next(m for m in order if m in final_eval["scores"] and final_eval["scores"][m] >= top - TIE_TOLERANCE)
ensemble_cv_auc = final_eval["scores"][ensemble_method]
print(f"Chosen ensemble method: {ensemble_method} | members: {final_members} | honest CV ROC-AUC = {ensemble_cv_auc:.6f}")


def apply_blend(pred_by_model):
    """Blend per-model prediction arrays (keys = model names) with the chosen, OOF-fitted method."""
    P = np.column_stack([pred_by_model[n] for n in final_members])
    if ensemble_method == "single":
        return pred_by_model[final_eval["best_single"]]
    if ensemble_method == "mean":
        return P.mean(1)
    if ensemble_method == "rank_mean":
        return np.column_stack([rank01(P[:, j]) for j in range(P.shape[1])]).mean(1)
    if ensemble_method == "weighted":
        return P @ final_eval["weights"]
    return final_eval["meta"].predict_proba(logit(np.clip(P, 1e-6, 1 - 1e-6)))[:, 1]

## 18. Final Model Selection

One final pipeline, assembled from the results above.

In [ ]:
best_single_name = max((n for n in cv_results if n != "baseline_lr"), key=lambda n: cv_results[n]["oof_auc"])
final_config = dict(
    features=features,
    feature_set=best_set,
    models={n: dict(family=cv_results[n]["family"], params=cv_results[n]["params"],
                    mean_best_iter=None if cv_results[n]["best_iters"][0] is None else float(np.mean(cv_results[n]["best_iters"])))
            for n in final_members},
    ensemble_method=ensemble_method,
    ensemble_weights=({n: round(float(w), 4) for n, w in zip(final_members, final_eval["weights"])}
                      if ensemble_method == "weighted" else None),
    cv=f"StratifiedKFold(n_splits={N_FOLDS}, shuffle=True, random_state={SEED}); OOF predictions",
    final_cv_auc=ensemble_cv_auc,
)
print(f"Feature set: '{final_config['feature_set']}' ({len(features)} features)\n{features}\n")
for n, spec in final_config["models"].items():
    print(f"{n}: family={spec['family']}, mean best iteration={spec['mean_best_iter']}\n    params={spec['params']}")
print(f"\nEnsemble: {ensemble_method} | weights: {final_config['ensemble_weights']}")
print(f"CV: {final_config['cv']}")
print(f"Final CV ROC-AUC (honest OOF estimate): {ensemble_cv_auc:.6f}")

## 19. Full Training

Each ensemble member is refit on **all** training rows. There is no validation fold left for early stopping, so boosters use the mean best iteration found in CV, scaled by `REFIT_SCALE` because the refit sees about 25% more data. As a sanity check, each full-data prediction is compared with that model's fold-averaged test prediction, which they should track very closely. No test labels exist or are used.

In [ ]:
REFIT_SCALE = 1.1


def fit_full_predict(family, params, mean_best_iter):
    X_data, X_new = MATRICES[family]
    n_rounds = None if mean_best_iter is None else max(50, int(mean_best_iter * REFIT_SCALE))
    if family == "lgb":
        booster = lgb.train({**LGB_FIXED, **params, "seed": SEED}, lgb.Dataset(X_data, y), n_rounds)
        return booster.predict(X_new)
    if family == "xgb":
        model = xgb.XGBClassifier(**{**XGB_FIXED, **params}, n_estimators=n_rounds, random_state=SEED).fit(X_data, y)
        return model.predict_proba(X_new)[:, 1]
    if family == "cat":
        model = CatBoostClassifier(**{**CAT_FIXED, **params}, iterations=n_rounds, random_seed=SEED)
        model.fit(Pool(X_data, y, cat_features=cat_features))
        return model.predict_proba(Pool(X_new, cat_features=cat_features))[:, 1]
    return ExtraTreesClassifier(**{**ET_FIXED, **params}, random_state=SEED).fit(X_data, y).predict_proba(X_new)[:, 1]


test_pred_full = {}
for name, spec in final_config["models"].items():
    test_pred_full[name] = fit_full_predict(spec["family"], spec["params"], spec["mean_best_iter"])
    corr = np.corrcoef(test_pred_full[name], cv_results[name]["test_avg"])[0, 1]
    print(f"{name:14s} full-data vs fold-average test predictions: Pearson r = {corr:.5f}")

## 20. Test Prediction

The full-data test predictions are blended with the method and weights fixed in section 17, which were fitted on OOF predictions only.

In [ ]:
final_test_pred = np.asarray(apply_blend(test_pred_full), dtype=float)
assert final_test_pred.shape == (len(test_df),)
print(pd.Series(final_test_pred, name="final_test_pred").describe().to_frame().T)
print(f"Mean predicted probability: {final_test_pred.mean():.4f} (training positive rate: {y.mean():.4f})")

## 21. Submission

In [ ]:
submission = pd.DataFrame({ID_COL: test_df[ID_COL].to_numpy(), TARGET_COL: final_test_pred})

assert len(submission) == len(test_df)
assert list(submission.columns) == ["id", "Will_Buy_EV"]
assert submission["Will_Buy_EV"].notna().all()
assert np.isfinite(submission["Will_Buy_EV"]).all()
assert submission["Will_Buy_EV"].between(0, 1).all()
assert submission["id"].equals(test_df["id"])

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
submission.head()

## 22. Final Results

In [ ]:
# Reference only: OOF scores printed by the earlier version of this notebook (saved cell output), same 5-fold protocol.
PREVIOUS_NOTEBOOK_CV = {"LightGBM": 0.941324, "XGBoost": 0.941542, "CatBoost": 0.941562, "Nelder-Mead blend": 0.941730}
PREVIOUS_PUBLIC_LB = 0.94138          # Kaggle public LB of the earlier submission, as reported by the notebook owner

best_single_auc = cv_results[best_single_name]["oof_auc"]
print("VALIDATION (CV ROC-AUC, 5-fold stratified OOF on train)")
print(f"  Baseline (logistic regression):  {baseline_auc:.6f}")
print(f"  Best individual model:           {best_single_auc:.6f}  ({best_single_name})")
print(f"  Final ensemble ({ensemble_method}):".ljust(35) + f"{ensemble_cv_auc:.6f}  (honest estimate)")
print(f"  Improvement over baseline:       {ensemble_cv_auc - baseline_auc:+.6f}")
print(f"  Change vs earlier notebook's blend ({PREVIOUS_NOTEBOOK_CV['Nelder-Mead blend']:.6f}): "
      f"{ensemble_cv_auc - PREVIOUS_NOTEBOOK_CV['Nelder-Mead blend']:+.6f}  (CV vs CV)")

print("\nSUBMISSION")
print(f"  File:            submission.csv")
print(f"  Rows / columns:  {len(submission):,} / {list(submission.columns)}")
print(f"  Missing values:  {int(submission.isna().sum().sum())}")
print(f"  Probability range: [{submission['Will_Buy_EV'].min():.6f}, {submission['Will_Buy_EV'].max():.6f}]")

print("\nKAGGLE PUBLIC LB ROC-AUC")
print(f"  Earlier submission: {PREVIOUS_PUBLIC_LB} (reported by the notebook owner).")
print("  This notebook's submission.csv has NOT been submitted yet, so no new public LB score is reported.")